#### CSC 296S Deep Learning (Spring 2026)

#### Dr. Haiquan Chen, Dept of Computer Scicence

#### California State University, Sacramento



## Imports & Functions

In [24]:
import os, csv, re
import librosa
import numpy as np
import soundfile as sf
import random
import pandas as pd
from tqdm import tqdm
import torch
import soundfile
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import librosa.display
import time

## Data Prep

In [25]:
#seed for data randomization, hard coded for reproducability
SEED = 534
torch.manual_seed(SEED) 
torch.backends.mps.deterministic = True 
torch.backends.mps.benchmark = False

SAMPLE_RATE = 44100
BATCH_SIZE = 32

In [26]:
#filepaths
test_dataframe_live = "../live_test/Test_Dataframe_Live.csv"
training_dataset_csv_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/Train_Dataset.csv"
testing_dataset_csv_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/Test_Dataset.csv"
#training datasets
train_sound_training_data_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/training/train_sounds"
urban_ambient_training_data_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/training/urban_ambient"
#testing datasets
train_sound_testing_data_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/testing/Train"
urban_ambient_testing_data_path = "../data/Datasets_TTA_Split_A_Weight_dB_SPL_2sec_Aug_Dataset/testing/Urban_Ambiente"

## Creating Randomized CSV File and Data Frames

In [27]:
#helper function, sorts audio files into a consistant order when building the csv
def extract_first_two_numbers(file_name):
    #check regex of file name
    match = re.match(r"(\d+)_(\d+)_", file_name)
    #return (38, 1)
    if match:
        first_num, second_num = int(match.group(1)), int(match.group(2))
        return first_num, second_num
    #if filename does not match pattern
    return float('inf'), float('inf')  

In [28]:
#create the CSV files for file accessability

#for training dataset
with open(training_dataset_csv_path, 'w+', newline='') as csv_file:
    writer = csv.writer(csv_file, delimiter=',')
    writer.writerow(['audio_name', 'label', 'path'])
    
    for path, dirs, files in os.walk(train_sound_training_data_path):
        sorted_files = sorted(files, key=extract_first_two_numbers)
 
        for file_name in sorted_files:
            if file_name.endswith('.wav'):
                full_path = os.path.normpath(os.path.join(path, file_name))
                writer.writerow([file_name, 'train_sound', full_path])

    for path, dirs, files in os.walk(urban_ambient_training_data_path):
        sorted_files = sorted(files, key=extract_first_two_numbers)
        
        for file_name in sorted_files:
            if file_name.endswith('.wav'):
                full_path = os.path.normpath(os.path.join(path, file_name))
                writer.writerow([file_name, 'urban_ambient', full_path])

#for testing dataset
with open(testing_dataset_csv_path, 'w+', newline='') as csv_file:
    writer = csv.writer(csv_file, delimiter=',')
    writer.writerow(['audio_name', 'label', 'path'])
    
    for path, dirs, files in os.walk(train_sound_testing_data_path):
        sorted_files = sorted(files, key=extract_first_two_numbers)
 
        for file_name in sorted_files:
            if file_name.endswith('.wav'):
                full_path = os.path.normpath(os.path.join(path, file_name))
                writer.writerow([file_name, 'train_sound', full_path])

    for path, dirs, files in os.walk(urban_ambient_testing_data_path):
        sorted_files = sorted(files, key=extract_first_two_numbers)
        
        for file_name in sorted_files:
            if file_name.endswith('.wav'):
                full_path = os.path.normpath(os.path.join(path, file_name))
                writer.writerow([file_name, 'urban_ambient', full_path])



In [29]:
#training dataset info
dataframe_training = pd.read_csv(training_dataset_csv_path) 
dataframe_training.tail()

,audio_name,label,path
28278,18_29_Amb_Verkehrslaermmessungen.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
28279,18_30_Amb_Verkehrslaermmessungen.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
28280,18_31_Amb_Verkehrslaermmessungen.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
28281,18_32_Amb_Verkehrslaermmessungen.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
28282,18_33_Amb_Verkehrslaermmessungen.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...


In [30]:
#testing dataset info
dataframe_testing = pd.read_csv(testing_dataset_csv_path) 
dataframe_testing.tail()

,audio_name,label,path
6239,1_25_Test_USM_traffic.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
6240,1_26_Test_USM_traffic.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
6241,1_27_Test_USM_traffic.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
6242,1_28_Test_USM_traffic.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...
6243,1_29_Test_USM_traffic.wav,urban_ambient,../data/Datasets_TTA_Split_A_Weight_dB_SPL_2se...


## Shuffle the Datasets Based on Random State Seed

In [31]:
dataframe_training = dataframe_training.sample(frac=1, random_state=SEED).reset_index(drop=False)
dataframe_testing = dataframe_testing.sample(frac=1, random_state=SEED).reset_index(drop=False)